# PixelDraft — Train the UI Detector on a Free GPU

This notebook trains the Phase-1 detector on Google Colab's free GPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

The plan:
1. Install Ultralytics
2. Get your Phase-0 dataset onto Colab (zip upload or Google Drive)
3. Train
4. Check metrics + visualize predictions
5. Download `best.pt` back to your machine


### 1. Install + confirm GPU

In [ ]:
!pip install ultralytics -q
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

### 2. Get your dataset onto Colab

**Option A — zip upload (simplest for a few thousand images).**
On your machine, zip the `dataset/` folder you made in Phase 0, then run the cell
and pick the zip.

In [ ]:
from google.colab import files
up = files.upload()          # choose dataset.zip
import zipfile, os
zname = list(up.keys())[0]
with zipfile.ZipFile(zname) as z:
    z.extractall("/content")
print("Extracted. Looking for data.yaml ...")
for root,_,fs in os.walk("/content"):
    if "data.yaml" in fs:
        DATA = os.path.join(root,"data.yaml"); print("Found:", DATA); break

**Option B — Google Drive (better if you'll retrain often).**
Put `dataset/` in your Drive, then mount it and set `DATA` to its `data.yaml`.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# DATA = '/content/drive/MyDrive/pixeldraft/dataset/data.yaml'

### 3. Fix the dataset path

Colab's path differs from your laptop's, so rewrite the `path:` line in data.yaml
to point at wherever it actually landed.

In [ ]:
import os, yaml
data_dir = os.path.dirname(DATA)
with open(DATA) as f: cfg = yaml.safe_load(f)
cfg['path'] = data_dir
with open(DATA,'w') as f: yaml.safe_dump(cfg, f)
print("data.yaml now points at:", data_dir)
print("classes:", cfg['nc'])

### 4. Train

`yolov8n` (nano) is fast and a fine baseline. Bump to `yolov8s` for more accuracy
if the GPU has headroom. `imgsz=960` matters — UI elements are small and you don't
want to lose checkboxes to downscaling.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')      # pretrained; downloads automatically on Colab
model.train(
    data=DATA, epochs=60, imgsz=960, batch=16,
    name='ui_detector', patience=15,
    fliplr=0.0, flipud=0.0, degrees=0.0, mosaic=0.0,   # UIs are orientation-fixed
    hsv_s=0.4, hsv_v=0.4, translate=0.05, scale=0.2,
)

### 5. Metrics + sample predictions

In [ ]:
m = model.val()
print("mAP50:   ", round(float(m.box.map50),3))
print("mAP50-95:", round(float(m.box.map),3))

In [ ]:
# visualize predictions on a few val images
import glob
from IPython.display import Image, display
val_imgs = glob.glob(os.path.join(data_dir,'images/val/*.png'))[:3]
for p in val_imgs:
    r = model(p, conf=0.25, verbose=False)[0]
    out = p.split('/')[-1]
    r.save(filename=out)
    display(Image(out))

### 6. Download your trained weights

In [ ]:
from google.colab import files
files.download('runs/detect/ui_detector/weights/best.pt')
# put this best.pt next to infer.py on your machine

---
**Next:** with `best.pt` on your machine, run
`python infer.py --weights best.pt --img some_screenshot.png --save out.png`
then move to **Phase 2** (boxes → layout tree → React).